In [1]:
import pandas as pd
import numpy as np
import os
import csv
import sys

In [4]:
baseline_path = r"C:\Users\nda84\Documents\DARe_project\Northumbria_Healthcare_Trust\MATSim_inputs\Baseline\MATSim_outputs\100.trips.csv"

In [8]:
df_baseline = pd.read_csv(baseline_path, index_col=None, header=0, delimiter=";")

In [9]:
df_baseline.head()

,person,trip_number,trip_id,dep_time,trav_time,wait_time,traveled_distance,euclidean_distance,main_mode,longest_distance_mode,...,start_facility_id,start_link,start_x,start_y,end_facility_id,end_link,end_x,end_y,first_pt_boarding_stop,last_pt_egress_stop
0,0_E02001768,1,0_E02001768_1,08:35:18,00:14:25,00:00:00,962,740,NaN,walk,...,NaN,5223734524244713139_5223734524307500581,436717.0,567137.0,NaN,5223734828042602893_5223734828052019507,437352.0,566757.0,NaN,NaN
1,0_E02001768,2,0_E02001768_2,15:24:21,00:14:25,00:00:00,962,740,NaN,walk,...,NaN,5223734828042602893_5223734828052019507,437352.0,566757.0,NaN,5223734524244713139_5223734524307500581,436717.0,567137.0,NaN,NaN
2,100000_E02001803,1,100000_E02001803_1,16:23:11,00:05:19,00:00:00,355,272,NaN,walk,...,NaN,5223725318315308501_5223725318576111137,438679.0,557158.0,NaN,5223722756428970499_5223722756477000671,438446.0,557016.0,NaN,NaN
3,100000_E02001803,2,100000_E02001803_2,17:45:14,00:05:19,00:00:00,355,272,NaN,walk,...,NaN,5223722756428970499_5223722756477000671,438446.0,557016.0,NaN,5223725318315308501_5223725318576111137,438679.0,557158.0,NaN,NaN
4,100001_E02001752,1,100001_E02001752_1,08:22:28,00:19:18,00:00:00,1288,990,NaN,walk,...,NaN,37629,433925.0,570002.0,NaN,5223734099242266979_5223734100729264173,434347.0,570898.0,NaN,NaN


In [10]:
df_baseline = df_baseline[['person', 'trip_id', 'trav_time', 'dep_time', 'traveled_distance', 'longest_distance_mode', 'start_activity_type', 'end_activity_type']]

In [11]:
df_baseline = df_baseline.rename(columns={'trav_time': 'B_trav_time', 'dep_time': 'B_dep_time', 'traveled_distance': 'B_traveled_distance', 
                        'longest_distance_mode': 'B_longest_distance_mode', 'start_activity_type': 'B_start_activity_type', 
                        'end_activity_type': 'B_end_activity_type'})

In [12]:
simulation_path = r"C:\Users\nda84\Documents\DARe_project\Northumbria_Healthcare_Trust\MATSim_inputs\RainfallScenario_CASE2_CurrentScenario\MATSim_outputs\500.trips.csv"

In [13]:
df_simulation = pd.read_csv(simulation_path, index_col=None, header=0, delimiter=";")

In [14]:
df_simulation = df_simulation[['trip_id', 'trav_time',  'dep_time', 'traveled_distance', 'longest_distance_mode', 'start_activity_type', 'end_activity_type']]

In [15]:
df_simulation = df_simulation.rename(columns={'trav_time': 'S_trav_time', 'dep_time': 'S_dep_time', 'traveled_distance': 'S_traveled_distance', 
                        'longest_distance_mode': 'S_longest_distance_mode', 'start_activity_type': 'S_start_activity_type', 
                        'end_activity_type': 'S_end_activity_type',})

In [16]:
def time_converter(x):
    h, m, s = x.split(":")
    return int(h) *3600 + int(m) * 60 + float(s)

In [17]:
df_baseline['B_trav_time_s'] = df_baseline['B_trav_time'].apply(time_converter)

In [18]:
df_baseline['B_dep_time'] = df_baseline['B_dep_time'].apply(time_converter)

In [19]:
df_simulation['S_trav_time_s'] = df_simulation['S_trav_time'].apply(time_converter)

In [20]:
df_simulation['S_dep_time'] = df_simulation['S_dep_time'].apply(time_converter)

In [21]:
df_comparison = pd.merge(df_baseline[['person', 'trip_id', 'B_trav_time_s', 'B_dep_time', 'B_traveled_distance', 'B_longest_distance_mode', 'B_start_activity_type', 'B_end_activity_type']], df_simulation[['trip_id', 'S_trav_time_s', 'S_dep_time', 'S_traveled_distance', 'S_longest_distance_mode']], how="left", on=["trip_id"])

In [22]:
df_comparison

,person,trip_id,B_trav_time_s,B_dep_time,B_traveled_distance,B_longest_distance_mode,B_start_activity_type,B_end_activity_type,S_trav_time_s,S_dep_time,S_traveled_distance,S_longest_distance_mode
0,0_E02001768,0_E02001768_1,865.0,30918.0,962,walk,home_30600,education_0_15_23400,785.0,30902.0,1352.0,pt
1,0_E02001768,0_E02001768_2,865.0,55461.0,962,walk,education_0_15_23400,home_29700,865.0,55596.0,962.0,walk
2,100000_E02001803,100000_E02001803_1,319.0,58991.0,355,walk,home_57600,supermarket_4500,319.0,58991.0,355.0,walk
3,100000_E02001803,100000_E02001803_2,319.0,63914.0,355,walk,supermarket_4500,home_21600,319.0,63914.0,355.0,walk
4,100001_E02001752,100001_E02001752_1,1158.0,30148.0,1288,walk,home_30600,education_0_15_31500,1158.0,30285.0,1288.0,walk
...,...,...,...,...,...,...,...,...,...,...,...,...
592276,NHS_3844,NHS_3844_1,1795.0,56657.0,16927,car,NHStaxi_origin,NHStaxi_destination,1776.0,55718.0,16927.0,car
592277,NHS_3845,NHS_3845_1,806.0,58465.0,10765,car,NHStaxi_origin,NHStaxi_destination,867.0,57284.0,10775.0,car
592278,NHS_3846,NHS_3846_1,1188.0,59241.0,13023,car,NHStaxi_origin,NHStaxi_destination,1178.0,58027.0,13023.0,car
592279,NHS_3847,NHS_3847_1,1027.0,60119.0,10765,car,NHStaxi_origin,NHStaxi_destination,1190.0,59203.0,10775.0,car


In [ ]:
######################################################################################################

In [ ]:
# Analyse time delays (all, morning peak (7-9AM), afternoon peak (4-6PM))

In [ ]:
df_comparison['Diff_trav_time_s'] = df_comparison['S_trav_time_s'] - df_comparison['B_trav_time_s']

In [ ]:
# Statistics about the global delay
#df_comparison['Diff_trav_time_s'].describe()

In [ ]:
# GLOBAL Delay in MINUTES
df_comparison['Diff_trav_time_s'].mean() / 60

In [ ]:
# Take only into account those trips delayed
df_comparison_TravelTime = df_comparison.loc[df_comparison['Diff_trav_time_s'] > 0]

In [ ]:
len(df_comparison_TravelTime)

In [ ]:
#df_comparison_TravelTime['Diff_trav_time_s'].describe()

In [ ]:
# AVERAGE trip delay (only considering delayed trips) in MINUTES
df_comparison_TravelTime['Diff_trav_time_s'].mean()/60

In [ ]:
# Percentage of delayed trips:
round(len(df_comparison_TravelTime)/len(df_baseline)* 100,2)

In [ ]:
morningPeak_start = 7*3600
morningPeak_end = 9*3600

afternoonPeak_start = 16*3600
afternoonPeak_end = 18*3600


In [ ]:
# AM peak delay (7-9h)   
df_comparison_TravelTime_delayAM = df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['B_dep_time'].between(morningPeak_start, morningPeak_end)))]

In [ ]:
df_comparison_TravelTime_delayAM['Diff_trav_time_s'].describe()

In [ ]:
df_comparison_TravelTime_delayAM['Diff_trav_time_s'].mean()/60

In [ ]:
# AM peak delay (7-9h)   
df_comparison_TravelTime_delayPM = df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['B_dep_time'].between(afternoonPeak_start, afternoonPeak_end)))]

In [ ]:
df_comparison_TravelTime_delayPM['Diff_trav_time_s'].describe()

In [ ]:
df_comparison_TravelTime_delayPM['Diff_trav_time_s'].mean() / 60

In [ ]:
# Analyse number of delayed trips (all, morning peak (7-9AM), afternoon peak (4-6PM))

In [ ]:
perc_total_delayed_trips = round(len(df_comparison_TravelTime)/len(df_baseline) * 100 ,2)

In [ ]:
perc_total_delayed_trips

In [ ]:
perc_AM_delayed_trips = round(len(df_comparison_TravelTime_delayAM)/ len(df_comparison.loc[df_comparison['B_dep_time'].between(morningPeak_start, morningPeak_end)]) * 100,2)

In [ ]:
perc_AM_delayed_trips

In [ ]:
perc_PM_delayed_trips = round(len(df_comparison_TravelTime_delayPM)/ len(df_comparison.loc[df_comparison['B_dep_time'].between(afternoonPeak_start, afternoonPeak_end)]) * 100,2)

In [ ]:
perc_PM_delayed_trips

In [ ]:
# Average delayed trips by trip purpose

In [ ]:
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("escort_work_"), "B_start_activity_type"] = "escort_work"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("escort_education"), "B_start_activity_type"] = "escort_education"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("escort_home"), "B_start_activity_type"] = "escort_home"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("escort_other"), "B_start_activity_type"] = "escort_other"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("eat"), "B_start_activity_type"] = "eat"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("education_0_15"), "B_start_activity_type"] = "education_0_15"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("education_16_18"), "B_start_activity_type"] = "education_16_18"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("education_18plus"), "B_start_activity_type"] = "education_18plus"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("home_"), "B_start_activity_type"] = "home"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("leisure_act"), "B_start_activity_type"] = "leisure_act"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("leisure_sport"), "B_start_activity_type"] = "leisure_sport"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("medical"), "B_start_activity_type"] = "medical"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("other"), "B_start_activity_type"] = "other"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("shop"), "B_start_activity_type"] = "shop"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("supermarket"), "B_start_activity_type"] = "supermarket"
df_comparison.loc[df_comparison["B_start_activity_type"].str.contains("work_"), "B_start_activity_type"] = "work"

In [ ]:
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("escort_work_"), "B_end_activity_type"] = "escort_work"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("escort_education"), "B_end_activity_type"] = "escort_education"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("escort_home"), "B_end_activity_type"] = "escort_home"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("escort_other"), "B_end_activity_type"] = "escort_other"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("eat"), "B_end_activity_type"] = "eat"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("education_0_15"), "B_end_activity_type"] = "education_0_15"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("education_16_18"), "B_end_activity_type"] = "education_16_18"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("education_18plus"), "B_end_activity_type"] = "education_18plus"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("home_"), "B_end_activity_type"] = "home"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("leisure_act"), "B_end_activity_type"] = "leisure_act"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("leisure_sport"), "B_end_activity_type"] = "leisure_sport"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("medical"), "B_end_activity_type"] = "medical"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("other"), "B_end_activity_type"] = "other"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("shop"), "B_end_activity_type"] = "shop"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("supermarket"), "B_end_activity_type"] = "supermarket"
df_comparison.loc[df_comparison["B_end_activity_type"].str.contains("work_"), "B_end_activity_type"] = "work"

In [ ]:
trip_purpose_list = df_comparison['B_end_activity_type'].unique().tolist()

In [ ]:
trip_purpose_list

In [ ]:
print('The average delay (minutes) per trip purpose is:')
for trip in trip_purpose_list:

    df = df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['B_end_activity_type'] == trip))]
    average_trip_delay = round(df['Diff_trav_time_s'].mean() / 60 ,2)
    print('  ** ' + trip + ': ' + str(average_trip_delay))
    
    

In [ ]:
print('The number and percentage of delayed trips per trip purpose is:')
for trip in trip_purpose_list:
    num = len(df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['B_end_activity_type'] == trip))])
    perc = round(num/len(df_comparison.loc[df_comparison['Diff_trav_time_s'] > 0])* 100, 2)
    total_perc = round(num/len(df_baseline) * 100, 2)
    print('  ** Trip to ' + trip + ':\t ' + str(num) + ',\t ' + str(perc) + '%,\t (' + str(total_perc) + '% of total simulated trips)')

In [ ]:
print('The number and percentage of delayed trips IN THE MORNING PEAK (7-9AM) per trip purpose is:')
for trip in trip_purpose_list:
    num = len(df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) &
                             (df_comparison['B_end_activity_type'] == trip) &
                             (df_comparison['B_dep_time'].between(morningPeak_start, morningPeak_end)))])
    
    perc = round(num/len(df_comparison.loc[(df_comparison['Diff_trav_time_s'] > 0) 
                         & df_comparison['B_dep_time'].between(morningPeak_start, morningPeak_end)])* 100, 2)
    
    total_perc = round(num/len(df_baseline.loc[df_baseline['B_dep_time'].between(morningPeak_start, morningPeak_end)]) * 100, 2)
    
    print('  ** Trip to ' + trip + ':\t ' + str(num) + ',\t ' + str(perc) + '%, \t (' + str(total_perc) + '% of total AM simulated trips)')

In [ ]:
print('The number and percentage of delayed trips IN THE AFTERNOON PEAK (4-6PM) per trip purpose is:')
for trip in trip_purpose_list:
    num = len(df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) &
                             (df_comparison['B_end_activity_type'] == trip) &
                             (df_comparison['B_dep_time'].between(afternoonPeak_start, afternoonPeak_end)))])
    
    perc = round(num/len(df_comparison.loc[(df_comparison['Diff_trav_time_s'] > 0) 
                         & df_comparison['B_dep_time'].between(afternoonPeak_start, afternoonPeak_end)])* 100, 2)
    
    total_perc = round(num/len(df_baseline.loc[df_baseline['B_dep_time'].between(afternoonPeak_start, afternoonPeak_end)]) * 100, 2)
    
    print('  ** Trip to ' + trip + ':\t ' + str(num) + ',\t ' + str(perc) + '%, \t (' + str(total_perc) + '% of total AM simulated trips)')

In [ ]:
transport_mode_list = df_comparison['B_longest_distance_mode'].unique().tolist()

In [ ]:
print('The number and percentage of delayed trips per transport mode is:')
for transport_mode in transport_mode_list:
    number = len(df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison["B_longest_distance_mode"] == transport_mode ))])
    percentage = round(number/len(df_comparison.loc[(df_comparison['Diff_trav_time_s'] > 0)]) * 100, 2)
    total_percentage = round(number/len(df_baseline) * 100, 2)
    print('  ** ' + transport_mode + ':\t ' + str(number) + ',\t ' + str(percentage)  + '%,\t (' + str(total_percentage) + '% of total trips)')

In [ ]:
###################################################################################################

In [ ]:
#Total delay in hours and days

In [ ]:
Total_delay_hours = int(df_comparison_TravelTime['Diff_trav_time_s'].sum()/3600)

In [ ]:
Total_delay_hours

In [ ]:
Total_delay_days = round(float(df_comparison_TravelTime['Diff_trav_time_s'].sum()/(3600 *24)),2)

In [ ]:
Total_delay_days

In [ ]:
###################################################################################################

In [ ]:
# Number of not finished trips --> cancelled

In [ ]:
total_cancelled_trips = len(df_comparison.loc[df_comparison['S_traveled_distance'].isnull()])

In [ ]:
total_cancelled_trips

In [ ]:
percentage_cancelled_trips = round(total_cancelled_trips/len(df_baseline) * 100,3)

In [ ]:
percentage_cancelled_trips

In [ ]:
print('The number and percentage of cancelled trips per trip purpose is:')
for trip in trip_purpose_list:
    number = len(df_comparison.loc[((df_comparison['S_traveled_distance'].isnull()) & (df_comparison["B_end_activity_type"] == trip ))])
    percentage = round(number/total_cancelled_trips * 100, 2)
    total_percentage = round(number/len(df_baseline) * 100, 2)
    print('  ** ' + trip + ':\t ' + str(number) + ',\t ' + str(percentage) + '%,\t (' + str(total_percentage) + '% of total trips)')

    

In [ ]:
transport_mode_list = df_comparison['B_longest_distance_mode'].unique().tolist()

In [ ]:
print('The number and percentage of cancelled trips per transport mode is:')
for transport_mode in transport_mode_list:
    number = len(df_comparison.loc[((df_comparison['S_traveled_distance'].isnull()) & (df_comparison["B_longest_distance_mode"] == transport_mode ))])
    percentage = round(number/total_cancelled_trips * 100, 2)
    total_percentage = round(number/len(df_baseline) * 100, 2)
    print('  ** ' + transport_mode + ':\t ' + str(number) + ',\t ' + str(percentage)  + '%,\t (' + str(total_percentage) + '% of total trips)')

In [ ]:
#####################################################################################################################

In [ ]:
# economic cost of delays depending on the trip purpose

In [ ]:
df_delayCost = df_comparison.loc[df_comparison['Diff_trav_time_s'] > 0]

In [ ]:
def unitCost_delay(row):
    
    if row['B_end_activity_type'] == 'work':
        return 12.86
    elif row['B_end_activity_type'] == 'escport_work':
        return 12.86
    elif row['B_end_activity_type'] == 'medical':
        return 9.39
    elif 'education' in row['B_end_activity_type']:
        return 7.63
    elif row['B_end_activity_type'] == 'home':
        return 5.87
    elif row['B_end_activity_type'] == 'escort_home':
        return 5.87
    elif row['B_end_activity_type'] == 'supermarket':
        return 5.87
    elif row['B_end_activity_type'] == 'shop':
        return 5.87
    elif row['B_end_activity_type'] == 'eat':
        return 5.87
    elif row['B_end_activity_type'] == 'leisure_sport':
        return 5.87
    elif row['B_end_activity_type'] == 'leisure_act':
        return 5.87
    elif row['B_end_activity_type'] == 'other':
        return 5.87
    else:
        return 5.87


In [ ]:
df_delayCost['Trip_delayUnit_cost'] = df_delayCost.apply(unitCost_delay, axis = 1)

In [ ]:
df_delayCost['delay_cost'] = round(df_delayCost['Diff_trav_time_s'] * df_delayCost['Trip_delayUnit_cost'] / 3600, 2)

In [ ]:
df_delayCost['delay_cost'].describe()

In [ ]:
df_delayCost['delay_cost'].sum()

In [ ]:
#######################################################################################################

In [ ]:
# economic cost of cancelled trips depending on the trip purpose

In [ ]:
df_CancellationCost = df_comparison.loc[df_comparison['S_traveled_distance'].isnull()]

In [ ]:
def unitCost_cancellation(row):
    
    if row['B_end_activity_type'] == 'work':
        return 102.88
    elif row['B_end_activity_type'] == 'escort_work':
        return 102.88
    elif row['B_end_activity_type'] == 'medical':
        return 75.14
    elif 'education' in row['B_end_activity_type']:
        return 61.05
    elif row['B_end_activity_type'] == 'home':
        return 46.96
    elif row['B_end_activity_type'] == 'escort_home':
        return 46.96
    elif row['B_end_activity_type'] == 'supermarket':
        return 0.0
    elif row['B_end_activity_type'] == 'shop':
        return 0.0
    elif row['B_end_activity_type'] == 'eat':
        return 0.0
    elif row['B_end_activity_type'] == 'leisure_sport':
        return 0.0
    elif row['B_end_activity_type'] == 'leisure_act':
        return 0.0
    elif row['B_end_activity_type'] == 'other':
        return 0.0
    else:
        return 0.0

In [ ]:
df_CancellationCost['Trip_cancellationUnit_cost'] = df_CancellationCost.apply(unitCost_cancellation, axis = 1)

In [ ]:
df_CancellationCost['Trip_cancellationUnit_cost'].sum()

In [ ]:
################################################################################################################

In [ ]:
# % Economic delay cost per trip purpose

In [ ]:
df_delayCost['B_end_activity_type'].unique()

In [ ]:
trip_purpose_list = df_delayCost['B_end_activity_type'].unique().tolist()

for trip_purpose in trip_purpose_list:

    df = df_delayCost.loc[df_delayCost['B_end_activity_type'] == trip_purpose]
    delay_cost_perTripPurpose = round(df['delay_cost'].sum(),2)
    perc = round(delay_cost_perTripPurpose/df_delayCost['delay_cost'].sum() *100,2)
    print('Trip to "' + trip_purpose + '" contributes the ' + str(perc) + '% of the trip delay cost')

    

In [ ]:
df_delayCost['delay_cost'].sum()

In [ ]:
################################################################################################################

In [ ]:
#  % Economic cancellation cost per trip purpose TBD

In [ ]:
#TBD

trip_purpose_list = df_CancellationCost['B_end_activity_type'].unique().tolist()

for trip_purpose in trip_purpose_list:

    df = df_CancellationCost.loc[df_CancellationCost['B_end_activity_type'] == trip_purpose]
    cancellation_cost_perTripPurpose = (df['Trip_cancellationUnit_cost'].sum())
    perc = round(cancellation_cost_perTripPurpose/df_CancellationCost['Trip_cancellationUnit_cost'].sum() *100,2)
    print('Trip to "' + trip_purpose + '" contributes to the ' + str(perc) + '% of the cancellation cost')

    

In [ ]:
df_CancellationCost['Trip_cancellationUnit_cost'].sum()

In [ ]:
################################################################################################################

In [ ]:
# Economic impact (delays + cancellations)
df_delayCost['delay_cost'].sum() + df_CancellationCost['Trip_cancellationUnit_cost'].sum()

In [ ]:
################################################################################################################

In [ ]:
# Modal shifts

In [ ]:
B_walk = len(df_comparison.loc[df_comparison['B_longest_distance_mode'] == 'walk'])
B_pt = len(df_comparison.loc[df_comparison['B_longest_distance_mode'] == 'pt'])
B_car = len(df_comparison.loc[df_comparison['B_longest_distance_mode'].str.contains('car')])
B_bike = len(df_comparison.loc[df_comparison['B_longest_distance_mode'] == 'bike'])
totaltrips = len(df_comparison)

B_walk_perc = round(B_walk/totaltrips *100,3)
B_pt_perc = round(B_pt/totaltrips *100,3)
B_car_perc = round(B_car/totaltrips *100,3)
B_bike_perc = round(B_bike/totaltrips *100,3)


df_non_cancelledTrips = df_comparison.loc[(~df_comparison['S_traveled_distance'].isnull())]

S_walk = len(df_non_cancelledTrips.loc[df_non_cancelledTrips['S_longest_distance_mode'] == 'walk'])
S_pt = len(df_non_cancelledTrips.loc[df_non_cancelledTrips['S_longest_distance_mode'] == 'pt'])
S_car = len(df_non_cancelledTrips.loc[df_non_cancelledTrips['S_longest_distance_mode'].str.contains("car")])
S_bike = len(df_non_cancelledTrips.loc[df_non_cancelledTrips['S_longest_distance_mode'] == 'bike'])

S_walk_perc = round(S_walk/totaltrips *100,3)
S_pt_perc = round(S_pt/totaltrips *100,3)
S_car_perc = round(S_car/totaltrips *100,3)
S_bike_perc = round(S_bike/totaltrips *100,3)

diff_walk_perc = S_walk_perc - B_walk_perc
diff_pt_perc = S_pt_perc - B_pt_perc
diff_car_perc = S_car_perc - B_car_perc
diff_bike_perc = S_bike_perc - B_bike_perc

print('The modal shift in Baseline is:\n  **Car: ' + str(B_car_perc) + '\n  **PT: ' + str(B_pt_perc) + '\n  **Walk: ' + str(B_walk_perc) + '\n  **Bike: ' + str(B_bike_perc) + '\n')
print('The modal shift in Scenario is:\n  **Car: ' + str(S_car_perc) + '\n  **PT: ' + str(S_pt_perc) + '\n  **Walk: ' + str(S_walk_perc) + '\n  **Bike: ' + str(S_bike_perc) + '\n')
print('The modal shift difference is:\n  **Car: ' + str(round(S_car_perc - B_car_perc, 3)) + '\n  **PT: ' + str(round(S_pt_perc - B_pt_perc,3)) + '\n  **Walk: ' + str(round(S_walk_perc - B_walk_perc, 3)) + '\n  **Bike: ' + str(round(S_bike_perc -B_bike_perc,3)) + '\n')

In [ ]:
S_walk_perc + S_pt_perc + S_car_perc + S_bike_perc + percentage_cancelled_trips

In [ ]:
B_walk_perc + B_pt_perc + B_car_perc + B_bike_perc

In [ ]:
round((S_car_perc - B_car_perc) + (S_pt_perc - B_pt_perc) + (S_walk_perc - B_walk_perc) + (S_bike_perc -B_bike_perc) + percentage_cancelled_trips, 3)

In [ ]:
percentage_cancelled_trips

In [ ]:
################################################################################################################

In [ ]:
# Social impact

In [ ]:
# Delays by socio-demographic attributes

In [ ]:
synthetic_pop_path = r"C:\Users\nda84\Documents\DARe_project\GARAGE\SynthPop_NE_2019.csv"

In [ ]:
df_syntheticPop = pd.read_csv(synthetic_pop_path, index_col=None, header=0, delimiter=",")

In [ ]:
df_syntheticPop

In [ ]:
simulated_agents_list = df_comparison['person'].unique().tolist()

In [ ]:
df_syntheticPopSimulated = df_syntheticPop.loc[df_syntheticPop['PID_AreaMSOA'].isin(simulated_agents_list)]

In [ ]:
len(df_syntheticPopSimulated)

In [ ]:
gender_type_list = df_syntheticPopSimulated['Sex'].unique().tolist()

In [ ]:
for gender in gender_type_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Sex'] == gender]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())
    
    if gender == 1.0:
        gender_type = 'males'
    else:
        gender_type = 'females'

    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('Number of ' + gender_type + ' experiencing delays is: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + gender_type +  ').' )



In [ ]:
df_syntheticPopSimulated['Occupation'] = df_syntheticPopSimulated['Occupation'].astype(str)

In [ ]:
Occupation_type_list = df_syntheticPopSimulated['Occupation'].unique().tolist()

In [ ]:
Occupation_type_list

In [ ]:
for occupation in Occupation_type_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Occupation'] == occupation]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())

    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('Number of agents whose occupation type "' + str(occupation) + '" experiencing delays is: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + str(occupation) +  ').' )



In [ ]:
age_range_list = [(0,16), (17,20), (21,29), (30,39), (40,49), (50,59), (60,64), (65, 74), (75,100)]

In [ ]:
for age_range in age_range_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Age'].between(age_range[0], age_range[1])]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())
    
    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('Number of agents between "' + str(age_range) + '" years old experiencing delays: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + str(age_range) +  ' age range).' )

    

In [ ]:
income_range_list = [(0,10000), (10001,20000),(20001,30000),(30001,40000),(40001,50000),(50001,60000), (60001,70000), (70001,80000), (80001,90000), (90001,5000000)]

In [ ]:
for income_range in income_range_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Income'].between(income_range[0], income_range[1])]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['Diff_trav_time_s'] > 0) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())
    
    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('The number of agent swith annual gross income between "' + str(income_range) + '"experiencing delays is: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + str(income_range) +  ' income range).' )

    

In [ ]:
# Cancellations by socio-demographic attributes

In [ ]:
for gender in gender_type_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Sex'] == gender]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['S_traveled_distance'].isnull()) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())
    
    if gender == 1.0:
        gender_type = 'males'
    else:
        gender_type = 'females'

    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('Number of ' + gender_type + ' experiencing trip cancellations is: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + gender_type +  ').' )



In [ ]:
for occupation in Occupation_type_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Occupation'] == occupation]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['S_traveled_distance'].isnull()) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())

    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('Number of agents whose occupation type "' + str(occupation) + '" experiencing trip cancellations is: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + str(occupation) +  ').' )



In [ ]:
for age_range in age_range_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Age'].between(age_range[0], age_range[1])]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['S_traveled_distance'].isnull()) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())
    
    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('Number of agents between "' + str(age_range) + '" years old experiencing trip cancellations: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + str(age_range) +  ' age range).' )



In [ ]:
for income_range in income_range_list:
    df = df_syntheticPopSimulated.loc[df_syntheticPopSimulated['Income'].between(income_range[0], income_range[1])]
    agents_list = df['PID_AreaMSOA'].unique().tolist()

    df_a = df_comparison.loc[((df_comparison['S_traveled_distance'].isnull()) & (df_comparison['person'].isin(agents_list)))]
    number_agents_affected = len(df_a['person'].unique())
    
    percentage_affected = round(number_agents_affected / len(df) *100,2)
    
    print('Nnumber of agent swith annual gross income between "' + str(income_range) + '"experiencing trip cancellation: ' + str(number_agents_affected) + ', (' + str(percentage_affected) + '% of total ' + str(income_range) +  ' income range).' )

 

In [ ]:
################################################################################################################